# LAM: Large Avatar Model - Colab Inference

`app_modal.py` / `lam_avatar_batch.py` と同じパイプラインを Modal なしで Colab 上で実行します。

**ソース**: `mirai-gpro/LAM_gpro` (lam-large-upload ブランチ)
**必要**: GPU ランタイム (T4 以上)
**Google Drive**: `/content/drive/MyDrive/LAM/` に model_zoo 等の重みファイルが必要

## 0. Google Drive マウント

In [ ]:
# ============================================================
# [0.1] Google Drive マウント
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_LAM = '/content/drive/MyDrive/LAM'
assert os.path.isdir(DRIVE_LAM), f'{DRIVE_LAM} not found!'
print('Drive LAM contents:')
!ls -la "$DRIVE_LAM"

## 1. 環境セットアップ (依存ライブラリ)

In [ ]:
# ============================================================
# [1.1] CUDA バージョン確認
# ============================================================
!nvcc --version
!nvidia-smi

In [ ]:
# ============================================================
# [1.2] システムパッケージ
# ============================================================
!apt-get update -qq
!apt-get install -y -qq libgl1-mesa-glx libglib2.0-0 ffmpeg ninja-build xz-utils > /dev/null 2>&1
print('System packages installed.')

In [ ]:
# ============================================================
# [1.3] PyTorch 確認 (Colab プリインストール版を使用)
# ============================================================
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ============================================================
# [1.4] numpy / scipy / xformers / chumpy
# ============================================================
!pip install -q --force-reinstall 'numpy>=1.26,<2' 'scipy>=1.11,<1.14'

# xformers: Colab プリインストール版を優先、なければ --no-deps で入れる
import subprocess, torch
ret = subprocess.run('python -c "import xformers; print(xformers.__version__)"',
                     shell=True, capture_output=True, text=True)
if ret.returncode == 0:
    print(f'xformers: {ret.stdout.strip()} (preinstalled)')
else:
    !pip install -q xformers --no-deps 2>/dev/null || echo 'xformers: install skipped (optional)'

!pip install -q chumpy==0.70 --no-build-isolation 2>/dev/null

# chumpy numpy 互換パッチ (numpy 2.x で削除された旧名を補う)
import importlib.util
chumpy_init = importlib.util.find_spec('chumpy').origin
!sed -i 's/from numpy import bool, int, float, complex, object, unicode, str, nan, inf/from numpy import nan, inf; import numpy; bool = numpy.bool_; int = numpy.int_; float = numpy.float64; complex = numpy.complex128; object = numpy.object_; unicode = numpy.str_; str = numpy.str_/' "$chumpy_init"
print(f'chumpy patched: {chumpy_init}')

# ★ numpy/scipy はここでは import しない (後の pip で上書きされるため [4.1] で初回 import する)

In [ ]:
# ============================================================
# [1.5] pytorch3d (Drive wheel キャッシュ対応)
# ============================================================
import subprocess, glob, os, torch

WHEEL_DIR = f'/content/drive/MyDrive/LAM/wheel_cache/torch{torch.version.cuda}'
os.makedirs(WHEEL_DIR, exist_ok=True)

# 1) プリビルトwheelを試す
ret = subprocess.run(
    'pip install -q --no-index --no-cache-dir pytorch3d '
    '-f https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/py310_cu121_pyt240/download.html',
    shell=True, capture_output=True
)
if ret.returncode == 0:
    print('pytorch3d installed (prebuilt wheel).')
else:
    # 2) Drive キャッシュを確認
    cached = sorted(glob.glob(f'{WHEEL_DIR}/pytorch3d-*.whl'))
    if cached:
        print(f'Installing from cache: {os.path.basename(cached[-1])}')
        subprocess.run(f'pip install -q {cached[-1]}', shell=True, check=True)
        print('pytorch3d installed (cached wheel).')
    else:
        # 3) ソースからビルドして wheel を保存
        print(f'Building pytorch3d from source (torch CUDA {torch.version.cuda}, 15-25 min)...')
        subprocess.run(
            'pip install -q git+https://github.com/facebookresearch/pytorch3d.git --no-build-isolation',
            shell=True, check=True
        )
        # ビルド済み wheel を Drive に保存
        result = subprocess.run('pip show pytorch3d', shell=True, capture_output=True, text=True)
        version = [l.split(':',1)[1].strip() for l in result.stdout.splitlines() if l.startswith('Version')][0]
        subprocess.run(
            f'pip wheel --no-deps --no-build-isolation pytorch3d=={version} -w {WHEEL_DIR}',
            shell=True
        )
        print(f'pytorch3d {version} installed & cached to Drive.')

In [ ]:
# ============================================================
# [1.6] その他の依存パッケージ + numpy 最終修正
# ============================================================
!pip install -q omegaconf==2.3.0 opencv-python-headless==4.9.0.80 \
  'imageio[ffmpeg]' moviepy==1.0.3 'rembg[gpu]' scikit-image pillow \
  'huggingface_hub>=0.24.0' transformers==4.44.2 diffusers==0.30.3 accelerate==0.34.2 \
  tyro==0.8.0 mediapipe==0.10.21 einops plyfile jaxtyping trimesh safetensors \
  loguru Cython PyMCubes typeguard rich decord ninja patool tensorboard

# onnxruntime-gpu (numpy 2.x を引き込むが後で修正)
!pip install -q onnxruntime-gpu==1.18.1 --extra-index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/

# ★ 全インストール後に numpy/scipy を正しいバージョンに戻す
!pip install -q --force-reinstall 'numpy>=1.26,<2' 'scipy>=1.11,<1.14'

# 検証 (subprocess = 新プロセスでディスク上のバージョンを確認)
import subprocess
_np_ver = subprocess.run('python -c "import numpy; print(numpy.__version__)"',
                         shell=True, capture_output=True, text=True).stdout.strip()
_sp_ver = subprocess.run('python -c "import scipy; print(scipy.__version__)"',
                         shell=True, capture_output=True, text=True).stdout.strip()
assert _np_ver.startswith('1.'), f'ERROR: numpy {_np_ver} on disk (expected 1.x)'
print(f'All pip packages installed. numpy={_np_ver}, scipy={_sp_ver}')

In [ ]:
# ============================================================
# [1.7] diff-gaussian-rasterization (Drive wheel キャッシュ対応)
# ============================================================
import subprocess, glob, os, torch

WHEEL_DIR = f'/content/drive/MyDrive/LAM/wheel_cache/torch{torch.version.cuda}'
os.makedirs(WHEEL_DIR, exist_ok=True)

cached = sorted(glob.glob(f'{WHEEL_DIR}/diff_gaussian_rasterization-*.whl'))
if cached:
    print(f'Installing from cache: {os.path.basename(cached[-1])}')
    subprocess.run(f'pip install -q {cached[-1]}', shell=True, check=True)
    print('diff-gaussian-rasterization installed (cached).')
else:
    print(f'Building diff-gaussian-rasterization from source (torch CUDA {torch.version.cuda})...')
    subprocess.run('rm -rf /tmp/dgr /tmp/dgr_wheel', shell=True)
    subprocess.run('git clone https://github.com/ashawkey/diff-gaussian-rasterization.git /tmp/dgr', shell=True)
    subprocess.run('git clone https://github.com/g-truc/glm.git /tmp/dgr/third_party/glm', shell=True)
    subprocess.run("find /tmp/dgr -name '*.cu' -exec sed -i '1i #include <cfloat>' {} +", shell=True)
    subprocess.run("find /tmp/dgr -name '*.h' -path '*/cuda_rasterizer/*' -exec sed -i '1i #include <cstdint>' {} +", shell=True)
    subprocess.run('pip wheel --no-deps --no-build-isolation /tmp/dgr -w /tmp/dgr_wheel', shell=True, check=True)
    # cache to Drive & install
    for whl in glob.glob('/tmp/dgr_wheel/*.whl'):
        subprocess.run(f'cp {whl} {WHEEL_DIR}/', shell=True)
        subprocess.run(f'pip install -q {whl}', shell=True, check=True)
    subprocess.run('rm -rf /tmp/dgr /tmp/dgr_wheel', shell=True)
    print('diff-gaussian-rasterization installed & cached to Drive.')

In [ ]:
# ============================================================
# [1.8] simple-knn (Drive wheel キャッシュ対応)
# ============================================================
import subprocess, glob, os, torch

WHEEL_DIR = f'/content/drive/MyDrive/LAM/wheel_cache/torch{torch.version.cuda}'
os.makedirs(WHEEL_DIR, exist_ok=True)

cached = sorted(glob.glob(f'{WHEEL_DIR}/simple_knn-*.whl'))
if cached:
    print(f'Installing from cache: {os.path.basename(cached[-1])}')
    subprocess.run(f'pip install -q {cached[-1]}', shell=True, check=True)
    print('simple-knn installed (cached).')
else:
    print(f'Building simple-knn from source (torch CUDA {torch.version.cuda})...')
    subprocess.run('rm -rf /tmp/simple-knn /tmp/sknn_wheel', shell=True)
    subprocess.run('git clone https://github.com/camenduru/simple-knn.git /tmp/simple-knn', shell=True)
    subprocess.run("sed -i '1i #include <cfloat>' /tmp/simple-knn/simple_knn.cu", shell=True)
    subprocess.run('pip wheel --no-deps --no-build-isolation /tmp/simple-knn -w /tmp/sknn_wheel', shell=True, check=True)
    # cache to Drive & install
    for whl in glob.glob('/tmp/sknn_wheel/*.whl'):
        subprocess.run(f'cp {whl} {WHEEL_DIR}/', shell=True)
        subprocess.run(f'pip install -q {whl}', shell=True, check=True)
    subprocess.run('rm -rf /tmp/simple-knn /tmp/sknn_wheel', shell=True)
    print('simple-knn installed & cached to Drive.')

In [ ]:
# ============================================================
# [1.9] nvdiffrast
# ============================================================
!pip install -q git+https://github.com/NVlabs/nvdiffrast.git --no-build-isolation
print('nvdiffrast installed.')

## 2. LAM リポジトリのクローンとセットアップ

In [ ]:
# ============================================================
# [2.1] LAM リポジトリクローン + cpu_nms コンパイル
# ============================================================
!git clone --depth 1 -b lam-large-upload https://github.com/mirai-gpro/LAM_gpro.git /content/LAM_tmp && \
  mv /content/LAM_tmp/LAM_Large_Avatar_Model /content/LAM && \
  rm -rf /content/LAM_tmp && echo "Done"

# cpu_nms コンパイル (numpy.int -> numpy.intp パッチ)
!sed -i 's/dtype=np\.int)/dtype=np.intp)/' /content/LAM/external/landmark_detection/FaceBoxesV2/utils/nms/cpu_nms.pyx

import subprocess
result = subprocess.run(
    ['python', '-c',
     'from setuptools import setup, Extension; '
     'from Cython.Build import cythonize; '
     'import numpy; '
     'setup(ext_modules=cythonize([Extension("cpu_nms", ["cpu_nms.pyx"])]), '
     'include_dirs=[numpy.get_include()])',
     'build_ext', '--inplace'],
    cwd='/content/LAM/external/landmark_detection/FaceBoxesV2/utils/nms',
    capture_output=True, text=True
)
print('cpu_nms build:', 'OK' if result.returncode == 0 else result.stderr[-200:])

In [ ]:
# ============================================================
# [2.2] torch.compile 無効化
# ============================================================
import os
for pattern in [
    '/content/LAM/lam/models/modeling_lam.py',
    '/content/LAM/lam/models/encoders/dinov2_fusion_wrapper.py',
    '/content/LAM/lam/losses/tvloss.py',
    '/content/LAM/lam/losses/pixelwise.py',
]:
    if os.path.isfile(pattern):
        !sed -i 's/^    @torch.compile$/    # @torch.compile  # DISABLED/' "$pattern"
        print(f'  Disabled torch.compile in {os.path.basename(pattern)}')
print('torch.compile disabled.')

In [ ]:
# ============================================================
# [2.3] Google Drive → LAM シンボリックリンク (model_zoo, assets)
# ============================================================
import os
import shutil

DRIVE_LAM = '/content/drive/MyDrive/LAM'
LAM_ROOT = '/content/LAM'

for subdir in ['model_zoo', 'assets', 'pretrained_models']:
    src = os.path.join(DRIVE_LAM, subdir)
    dst = os.path.join(LAM_ROOT, subdir)
    if not os.path.isdir(src):
        print(f'  SKIP: {src} not found')
        continue
    if os.path.islink(dst):
        os.unlink(dst)
    elif os.path.isdir(dst):
        shutil.rmtree(dst)
    os.symlink(src, dst)
    print(f'  Linked: {dst} -> {src}')

# pretrained_models/human_model_files -> model_zoo/human_parametric_models
pretrained_hm = os.path.join(LAM_ROOT, 'pretrained_models', 'human_model_files')
model_zoo_hpm = os.path.join(LAM_ROOT, 'model_zoo', 'human_parametric_models')
if not os.path.exists(pretrained_hm) and os.path.isdir(model_zoo_hpm):
    os.makedirs(os.path.dirname(pretrained_hm), exist_ok=True)
    os.symlink(model_zoo_hpm, pretrained_hm)
    print(f'  Linked: {pretrained_hm} -> {model_zoo_hpm}')

print('\nDrive model_zoo contents:')
!ls "$DRIVE_LAM/model_zoo/" 2>/dev/null || echo 'model_zoo not found'
print('\nDrive assets contents:')
!ls "$DRIVE_LAM/assets/" 2>/dev/null || echo 'assets not found'

## 3. 入力画像のアップロード

In [ ]:
# ============================================================
# [3.1] 画像アップロード
# ============================================================
from google.colab import files
from IPython.display import display, Image as IPImage

uploaded = files.upload()
INPUT_IMAGE = list(uploaded.keys())[0]
INPUT_PATH = f'/content/{INPUT_IMAGE}'

# 入力画像を表示
display(IPImage(filename=INPUT_PATH, width=300))
print(f'Input image: {INPUT_PATH} ({os.path.getsize(INPUT_PATH)} bytes)')

## 4. LAM パイプライン初期化 (公式 app_hf_space.py 準拠)

In [ ]:
# ============================================================
# [4.1] import + パス設定 + torch.compile 無効化
# ============================================================
import sys
import os
import argparse

import numpy as np
import torch
from PIL import Image
from omegaconf import OmegaConf

# torch.compile を完全無効化
try:
    import torch._dynamo
    torch._dynamo.config.disable = True
    torch._dynamo.reset()
except (ImportError, AttributeError):
    print('torch._dynamo not available, skipping (OK)')

_orig_compile = torch.compile
def _noop_compile(fn=None, *a, **kw):
    return fn if fn is not None else (lambda f: f)
torch.compile = _noop_compile

# パス設定
LAM_ROOT = '/content/LAM'
os.chdir(LAM_ROOT)
sys.path.insert(0, LAM_ROOT)
# ModelScope版: flame_tracking等はルート直下 (tools/ ではない)
if os.path.isdir(os.path.join(LAM_ROOT, 'tools')):
    sys.path.insert(0, os.path.join(LAM_ROOT, 'tools'))

print(f'Working dir: {os.getcwd()}')
print(f'numpy: {np.__version__}')
print(f'CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')

In [ ]:
# ============================================================
# [4.2] モデルパス設定 (環境変数)
# ============================================================
# --- パス: Modal版は model_zoo/ 配下、HF Space版は exps/ 配下 ---
# Google Drive の構造に合わせて設定してください
MODEL_NAME = './model_zoo/lam_models/releases/lam/lam-20k/step_045500/'
INFER_CONFIG = './configs/inference/lam-20k-8gpu.yaml'

# model_name パスが存在するか確認
if not os.path.isdir(MODEL_NAME):
    # HF Space 版のパスも試す
    alt = './exps/releases/lam/lam-20k/step_045500/'
    if os.path.isdir(alt):
        MODEL_NAME = alt
        print(f'Using HF Space model path: {MODEL_NAME}')
    else:
        print(f'WARNING: Model not found at {MODEL_NAME} or {alt}')
        print('Available in model_zoo:')
        !find ./model_zoo -name 'step_*' -type d 2>/dev/null | head -5
        print('Available in exps:')
        !find ./exps -name 'step_*' -type d 2>/dev/null | head -5

os.environ.update({
    'APP_ENABLED': '1',
    'APP_MODEL_NAME': MODEL_NAME,
    'APP_INFER': INFER_CONFIG,
    'APP_TYPE': 'infer.lam',
    'NUMBA_THREADING_LAYER': 'forksafe',
})

print(f'Model: {MODEL_NAME}')
print(f'Config: {INFER_CONFIG}')
print(f'Model dir exists: {os.path.isdir(MODEL_NAME)}')

In [ ]:
# ============================================================
# [4.3] parse_configs (公式そのまま)
# ============================================================
def parse_configs():
    parser = argparse.ArgumentParser()
    parser.add_argument('--config', type=str)
    parser.add_argument('--infer', type=str)
    args, unknown = parser.parse_known_args([])

    cfg = OmegaConf.create()
    cli_cfg = OmegaConf.from_cli([])

    if os.environ.get('APP_INFER') is not None:
        args.infer = os.environ.get('APP_INFER')
    if os.environ.get('APP_MODEL_NAME') is not None:
        cli_cfg.model_name = os.environ.get('APP_MODEL_NAME')

    args.config = args.infer if args.config is None else args.config

    if args.config is not None:
        cfg_train = OmegaConf.load(args.config)
        cfg.source_size = cfg_train.dataset.source_image_res
        try:
            cfg.src_head_size = cfg_train.dataset.src_head_size
        except:
            cfg.src_head_size = 112
        cfg.render_size = cfg_train.dataset.render_image.high
        _relative_path = os.path.join(
            cfg_train.experiment.parent,
            cfg_train.experiment.child,
            os.path.basename(cli_cfg.model_name).split('_')[-1],
        )
        cfg.save_tmp_dump = os.path.join('exps', 'save_tmp', _relative_path)
        cfg.image_dump = os.path.join('exps', 'images', _relative_path)
        cfg.video_dump = os.path.join('exps', 'videos', _relative_path)

    if args.infer is not None:
        cfg_infer = OmegaConf.load(args.infer)
        cfg.merge_with(cfg_infer)
        cfg.setdefault('save_tmp_dump', os.path.join('exps', cli_cfg.model_name, 'save_tmp'))
        cfg.setdefault('image_dump', os.path.join('exps', cli_cfg.model_name, 'images'))
        cfg.setdefault('video_dump', os.path.join('dumps', cli_cfg.model_name, 'videos'))
        cfg.setdefault('mesh_dump', os.path.join('dumps', cli_cfg.model_name, 'meshes'))

    cfg.motion_video_read_fps = 30
    cfg.merge_with(cli_cfg)
    cfg.setdefault('logger', 'INFO')
    assert cfg.model_name is not None, 'model_name is required'
    return cfg, cfg_train

cfg, cfg_train = parse_configs()
print(f'source_size: {cfg.source_size}')
print(f'render_size: {cfg.render_size}')
print(f'model_name: {cfg.model_name}')

In [ ]:
# ============================================================
# [4.4] LAM モデルのビルド (from_pretrained)
# ============================================================
import sys, os
LAM_ROOT = '/content/LAM'
if LAM_ROOT not in sys.path:
    sys.path.insert(0, LAM_ROOT)
    os.chdir(LAM_ROOT)
    print(f'sys.path fixed: added {LAM_ROOT}')

# 依存セルのチェック
assert 'cfg' in dir(), (
    'ERROR: cfg が未定義です。先に [4.1] → [4.2] → [4.3] を順番に実行してください。'
)

from lam.models import model_dict
from lam.utils.hf_hub import wrap_model_hub

hf_model_cls = wrap_model_hub(model_dict['lam'])
lam = hf_model_cls.from_pretrained(cfg.model_name)
lam.to('cuda')
print('LAM model loaded and moved to CUDA.')

In [ ]:
# ============================================================
# [4.5] FLAME tracking 初期化
# ============================================================
from flame_tracking_single_image import FlameTrackingSingleImage

# パスは Google Drive の構造に合わせて設定
# Modal版: model_zoo/flame_tracking_models/
# HF Space版: pretrain_model/
FLAME_BASE = './model_zoo/flame_tracking_models'
if not os.path.isdir(FLAME_BASE):
    FLAME_BASE = './pretrain_model'

flametracking = FlameTrackingSingleImage(
    output_dir='tracking_output',
    alignment_model_path=f'{FLAME_BASE}/68_keypoints_model.pkl',
    vgghead_model_path=f'{FLAME_BASE}/vgghead/vgg_heads_l.trcd',
    human_matting_path=f'{FLAME_BASE}/matting/stylematte_synth.pt',
    facebox_model_path=f'{FLAME_BASE}/FaceBoxesV2.pth',
    detect_iris_landmarks=False,  # Modal版と同じ (HF Space版はTrue)
)
print('FLAME tracking initialized.')

## 5. 推論実行 (公式 core_fn 準拠)

In [ ]:
# ============================================================
# [5.1] 準備: import + 画像保存 + モーション解決
# ============================================================
import sys, os, shutil, tempfile

LAM_ROOT = '/content/LAM'
if LAM_ROOT not in sys.path:
    sys.path.insert(0, LAM_ROOT)
    os.chdir(LAM_ROOT)
    print(f'sys.path fixed: added {LAM_ROOT}')

from lam.runners.infer.head_utils import prepare_motion_seqs, preprocess_image

# === パラメータ設定 ===
MOTION_NAME = 'GEM'  # 変更可能: Speeding_Scandal, Look_In_My_Eyes, 等

# 作業ディレクトリ
working_dir = tempfile.mkdtemp(prefix='lam_colab_')
print(f'Working dir: {working_dir}')

# 古い tracking データをクリア
for subdir in ['preprocess', 'tracking', 'export']:
    stale = os.path.join('tracking_output', subdir)
    if os.path.isdir(stale):
        shutil.rmtree(stale)
        print(f'  Cleaned: {stale}')

# Step 1: 入力画像を保存
print('\n[Step 1/5] Saving input image...')
image_raw = os.path.join(working_dir, 'raw.png')
with Image.open(INPUT_PATH).convert('RGB') as img:
    img.save(image_raw)
print(f'  Saved: {image_raw}')

# Step 2: モーション解決
print(f'\n[Step 2/5] Resolving motion: {MOTION_NAME}...')
flame_params_dir = f'./assets/sample_motion/export/{MOTION_NAME}/flame_param'
if not os.path.isdir(flame_params_dir):
    flame_params_dir = f'./model_zoo/sample_motion/export/{MOTION_NAME}/flame_param'
assert os.path.isdir(flame_params_dir), f'Motion not found: {flame_params_dir}'
print(f'  Using: {flame_params_dir}')

In [ ]:
# ============================================================
# [5.2] FLAME tracking (Step 3/5)
# ============================================================
print('[Step 3/5] FLAME tracking...')
return_code = flametracking.preprocess(image_raw)
assert return_code == 0, 'flametracking preprocess failed!'

return_code = flametracking.optimize()
assert return_code == 0, 'flametracking optimize failed!'

return_code, output_dir = flametracking.export()
assert return_code == 0, 'flametracking export failed!'

image_path = os.path.join(output_dir, 'images/00000_00.png')
mask_path = image_path.replace('/images/', '/fg_masks/').replace('.jpg', '.png')
print(f'  image_path: {image_path}')
print(f'  mask_path: {mask_path}')
print(f'  Both exist: {os.path.isfile(image_path) and os.path.isfile(mask_path)}')

In [ ]:
# ============================================================
# [5.3] Preprocess + Motion (Step 4/5)
# ============================================================
print('[Step 4/5] Preprocessing and preparing motion...')

aspect_standard = 1.0 / 1.0
source_size = cfg.source_size
render_size = cfg.render_size
render_fps = 30

motion_img_need_mask = cfg.get('motion_img_need_mask', False)
vis_motion = cfg.get('vis_motion', False)

# preprocess_image (公式と同じ引数)
image_tensor, _, _, shape_param = preprocess_image(
    image_path, mask_path=mask_path, intr=None, pad_ratio=0,
    bg_color=1., max_tgt_size=None, aspect_standard=aspect_standard,
    enlarge_ratio=[1.0, 1.0],
    render_tgt_size=source_size, multiply=14, need_mask=True, get_shape_param=True,
)

# shape_param チェック
sp = shape_param.detach().cpu().numpy()
print(f'  shape_param range: [{sp.min():.3f}, {sp.max():.3f}]')
print(f'  shape_param has NaN: {np.isnan(sp).any()}')
print(f'  shape_param max abs: {np.abs(sp).max():.3f}')

# 前処理画像を保存して表示
vis_ref_img = (image_tensor[0].permute(1, 2, 0).cpu().detach().numpy() * 255).astype(np.uint8)
ref_save_path = os.path.join(working_dir, 'preprocessed.png')
Image.fromarray(vis_ref_img).save(ref_save_path)

from IPython.display import display, Image as IPImage
print('\nPreprocessed image:')
display(IPImage(filename=ref_save_path, width=256))

In [ ]:
# ============================================================
# [5.4] prepare_motion_seqs
# ============================================================
motion_seqs_dir = flame_params_dir
src = image_path.split('/')[-3]
driven = motion_seqs_dir.split('/')[-2]
src_driven = [src, driven]

motion_seq = prepare_motion_seqs(
    motion_seqs_dir, None, save_root=working_dir, fps=render_fps,
    bg_color=1., aspect_standard=aspect_standard,
    enlarge_ratio=[1.0, 1, 0],
    render_image_res=render_size, multiply=16,
    need_mask=motion_img_need_mask, vis_motion=vis_motion,
    shape_param=shape_param, test_sample=False, cross_id=False,
    src_driven=src_driven,
)

print(f'  motion_seq keys: {list(motion_seq.keys())}')
print(f'  render_c2ws shape: {motion_seq["render_c2ws"].shape}')
print(f'  flame_params keys: {list(motion_seq["flame_params"].keys())}')

In [ ]:
# ============================================================
# [5.5] LAM inference (Step 5/5)
# ============================================================
print('[Step 5/5] LAM inference...')

motion_seq['flame_params']['betas'] = shape_param.unsqueeze(0)
device, dtype = 'cuda', torch.float32

print('start to inference...................')
with torch.no_grad():
    res = lam.infer_single_view(
        image_tensor.unsqueeze(0).to(device, dtype), None, None,
        render_c2ws=motion_seq['render_c2ws'].to(device),
        render_intrs=motion_seq['render_intrs'].to(device),
        render_bg_colors=motion_seq['render_bg_colors'].to(device),
        flame_params={k: v.to(device) for k, v in motion_seq['flame_params'].items()},
    )

print('Inference complete!')
print(f'  comp_rgb shape: {res["comp_rgb"].shape}')
print(f'  comp_mask shape: {res["comp_mask"].shape}')

## 6. 結果の表示

In [ ]:
# ============================================================
# [6.1] RGB 後処理
# ============================================================
rgb = res['comp_rgb'].detach().cpu().numpy()  # [Nv, H, W, 3]
mask = res['comp_mask'].detach().cpu().numpy()  # [Nv, H, W, 3]
mask[mask < 0.5] = 0.0
rgb = rgb * mask + (1 - mask) * 1
rgb = (np.clip(rgb, 0, 1.0) * 255).astype(np.uint8)

print(f'Output frames: {rgb.shape[0]}')
print(f'Frame size: {rgb.shape[1]}x{rgb.shape[2]}')

# 最初のフレームを表示
from IPython.display import display, Image as IPImage
preview_path = os.path.join(working_dir, 'preview.png')
Image.fromarray(rgb[0]).save(preview_path)
print('\nFirst frame:')
display(IPImage(filename=preview_path, width=300))

# 比較画像
img_in = Image.open(INPUT_PATH).convert('RGB').resize((256, 256))
img_out = Image.open(preview_path).convert('RGB').resize((256, 256))
canvas = Image.new('RGB', (512, 256), (255, 255, 255))
canvas.paste(img_in, (0, 0))
canvas.paste(img_out, (256, 0))
compare_path = os.path.join(working_dir, 'compare.png')
canvas.save(compare_path)
print('\nInput vs Output:')
display(IPImage(filename=compare_path, width=512))

In [ ]:
# ============================================================
# [6.2] 動画生成
# ============================================================
from moviepy.editor import ImageSequenceClip, VideoFileClip, AudioFileClip

video_path = os.path.join(working_dir, 'output.mp4')
images = [frame.astype(np.uint8) for frame in rgb]
clip = ImageSequenceClip(images, fps=render_fps)
clip.write_videofile(video_path, codec='libx264', logger=None)
print(f'Video saved: {video_path}')

# オーディオ追加
audio_path = f'./assets/sample_motion/export/{MOTION_NAME}/{MOTION_NAME}.wav'
if not os.path.isfile(audio_path):
    audio_path = f'./model_zoo/sample_motion/export/{MOTION_NAME}/{MOTION_NAME}.wav'

if os.path.isfile(audio_path):
    final_path = os.path.join(working_dir, 'output_audio.mp4')
    vc = VideoFileClip(video_path)
    ac = AudioFileClip(audio_path)
    if ac.duration > 10:
        ac = ac.subclip(0, 10)
    vc.set_audio(ac).write_videofile(final_path, codec='libx264', audio_codec='aac', logger=None)
    print(f'Video with audio: {final_path}')
else:
    final_path = video_path
    print(f'No audio found at {audio_path}')

# Colab で動画表示
from IPython.display import HTML
from base64 import b64encode
with open(final_path, 'rb') as f:
    mp4_b64 = b64encode(f.read()).decode()
display(HTML(f'<video controls width="400"><source src="data:video/mp4;base64,{mp4_b64}" type="video/mp4"></video>'))

In [ ]:
# ============================================================
# [6.3] 結果をダウンロード
# ============================================================
from google.colab import files

print('Downloading results...')
files.download(final_path)
files.download(compare_path)

## (オプション) デバッグ: 中間データ確認

In [ ]:
# ============================================================
# [Debug] FLAME tracking 結果の確認
# ============================================================
print('=== Tracking output ===')
!ls -la tracking_output/export/raw/images/ 2>/dev/null
!ls -la tracking_output/export/raw/fg_masks/ 2>/dev/null

# tracked image と mask を表示
from IPython.display import display, Image as IPImage
if os.path.isfile(image_path):
    print('\nTracked image:')
    display(IPImage(filename=image_path, width=256))
if os.path.isfile(mask_path):
    print('\nMask:')
    display(IPImage(filename=mask_path, width=256))

print('\n=== shape_param details ===')
print(f'shape_param shape: {shape_param.shape}')
print(f'shape_param dtype: {shape_param.dtype}')
print(f'shape_param values: {shape_param}')

print('\n=== motion_seq details ===')
for k, v in motion_seq.items():
    if hasattr(v, 'shape'):
        print(f'  {k}: shape={v.shape}, dtype={v.dtype}')
    elif isinstance(v, dict):
        print(f'  {k}:')
        for kk, vv in v.items():
            if hasattr(vv, 'shape'):
                print(f'    {kk}: shape={vv.shape}, dtype={vv.dtype}')